# Lab Assignment 3 - Part D: Physiological Signal Data
## MIT-BIH Arrhythmia Database (ECG)

Pipeline: acquire -> inspect -> noise removal (baseline wander, powerline,
high-frequency) -> EDA -> R-peak alignment -> beat segmentation into windows
-> AAMI class mapping -> feature extraction -> augmentation -> signal tensors

Dataset: MIT-BIH Arrhythmia Database (PhysioNet)
Download:  wget -r -N -c -np https://physionet.org/files/mitdb/1.0.0/
  or:      pip install wfdb  then use wfdb.dl_database('mitdb', 'data/mitdb')

In [ ]:
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as sps
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data/mitdb")
OUT_DIR = Path("outputs/partD")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 360               # MIT-BIH sampling frequency in Hz
WINDOW_BEFORE = 0.25   # seconds before the R peak
WINDOW_AFTER = 0.45    # seconds after the R peak
WIN_LEN = int((WINDOW_BEFORE + WINDOW_AFTER) * FS)   # 252 samples per beat

## D0. Download helper

In [ ]:
def download_mitdb(dest: Path = DATA_DIR) -> None:
    """Fetch the MIT-BIH database via the wfdb package."""
    import wfdb
    dest.mkdir(parents=True, exist_ok=True)
    wfdb.dl_database("mitdb", str(dest))
    print(f"Downloaded to {dest}")


# Uncomment on first run:
# download_mitdb()

## D1. Acquisition and inspection

In [ ]:
import wfdb   # noqa: E402

RECORDS = [
    "100", "101", "102", "103", "104", "105", "106", "107", "108", "109",
    "111", "112", "113", "114", "115", "116", "117", "118", "119", "121",
    "122", "123", "124", "200", "201", "202", "203", "205", "207", "208",
    "209", "210", "212", "213", "214", "215", "217", "219", "220", "221",
    "222", "223", "228", "230", "231", "232", "233", "234",
]

# Use a subset for a fast run; set to RECORDS for the full database
ACTIVE_RECORDS = RECORDS[:12]


def load_record(rec_id: str, data_dir: Path = DATA_DIR):
    """Read one record's signal and beat annotations."""
    path = str(data_dir / rec_id)
    record = wfdb.rdrecord(path)
    annotation = wfdb.rdann(path, "atr")
    return record, annotation


rec, ann = load_record(ACTIVE_RECORDS[0])
print("Record:", ACTIVE_RECORDS[0])
print("  signal shape:", rec.p_signal.shape)
print("  sampling frequency:", rec.fs, "Hz")
print("  lead names:", rec.sig_name)
print("  units:", rec.units)
print("  duration:", rec.p_signal.shape[0] / rec.fs / 60, "minutes")
print("  annotations:", len(ann.sample))
print("  annotation symbols:", Counter(ann.symbol).most_common(10))

## D2. AAMI class mapping

MIT-BIH uses ~19 raw beat symbols. The AAMI EC57 standard collapses these into
5 clinically meaningful superclasses, which is what arrhythmia classifiers are
actually evaluated against. Non-beat annotations (rhythm markers, signal
quality flags) are excluded - they are not beats and must not become training
samples.

In [ ]:
AAMI_MAP = {
    # N - normal and bundle branch block beats
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    # S - supraventricular ectopic
    "A": "S", "a": "S", "J": "S", "S": "S",
    # V - ventricular ectopic
    "V": "V", "E": "V",
    # F - fusion
    "F": "F",
    # Q - unclassifiable / paced
    "/": "Q", "f": "Q", "Q": "Q",
}
AAMI_CLASSES = ["N", "S", "V", "F", "Q"]

## D3. Noise removal

ECG carries three characteristic artefacts, each needing a different filter:

1. **Baseline wander** (below ~0.5 Hz) from breathing and electrode motion.
   Removed with a high-pass filter, or a two-stage median filter which is
   gentler on the ST segment - clinically important, since ST elevation is the
   marker of myocardial infarction and an aggressive high-pass distorts it.
2. **Powerline interference** at 50/60 Hz from mains electricity. Removed with
   a narrow notch filter.
3. **Muscle/EMG noise** at high frequency. Removed with a low-pass at 40 Hz.

Filtering uses `filtfilt` (zero-phase, applied forwards and backwards) rather
than `lfilter`, because a phase shift would move the R peak and corrupt the
QRS timing that every downstream measurement depends on.

In [ ]:
def remove_baseline_median(sig: np.ndarray, fs: int = FS) -> np.ndarray:
    """Two-stage median filter baseline removal (preserves ST segment)."""
    w1 = int(0.2 * fs) | 1     # 200 ms - removes QRS complexes
    w2 = int(0.6 * fs) | 1     # 600 ms - removes P and T waves
    baseline = sps.medfilt(sps.medfilt(sig, w1), w2)
    return sig - baseline


def bandpass_filter(sig: np.ndarray, low=0.5, high=40.0,
                    fs: int = FS, order: int = 4) -> np.ndarray:
    nyq = fs / 2
    b, a = sps.butter(order, [low / nyq, high / nyq], btype="band")
    return sps.filtfilt(b, a, sig)


def notch_filter(sig: np.ndarray, freq: float = 60.0,
                 fs: int = FS, q: float = 30.0) -> np.ndarray:
    b, a = sps.iirnotch(freq / (fs / 2), q)
    return sps.filtfilt(b, a, sig)


def denoise_ecg(sig: np.ndarray, fs: int = FS, powerline: float = 60.0) -> np.ndarray:
    out = remove_baseline_median(sig, fs)
    out = notch_filter(out, powerline, fs)
    out = bandpass_filter(out, 0.5, 40.0, fs)
    return out


raw_sig = rec.p_signal[:, 0]           # lead MLII
clean_sig = denoise_ecg(raw_sig)

# Visualise each stage on a 5-second strip
seg = slice(0, 5 * FS)
t = np.arange(5 * FS) / FS

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(t, raw_sig[seg]); axes[0].set_title("Raw ECG (lead MLII)")
axes[1].plot(t, remove_baseline_median(raw_sig)[seg], color="tab:orange")
axes[1].set_title("After baseline wander removal (median filter)")
axes[2].plot(t, notch_filter(remove_baseline_median(raw_sig))[seg], color="tab:green")
axes[2].set_title("After 60 Hz notch filter")
axes[3].plot(t, clean_sig[seg], color="tab:red")
axes[3].set_title("After 0.5-40 Hz bandpass (final)")
axes[3].set_xlabel("Time (s)")
for ax in axes:
    ax.set_ylabel("mV")
plt.tight_layout()
plt.savefig(OUT_DIR / "denoising_stages.png", dpi=120)
plt.close()

# Frequency-domain confirmation
f_raw, p_raw = sps.welch(raw_sig, FS, nperseg=2048)
f_cln, p_cln = sps.welch(clean_sig, FS, nperseg=2048)

fig, ax = plt.subplots(figsize=(11, 4))
ax.semilogy(f_raw, p_raw, label="raw", alpha=0.8)
ax.semilogy(f_cln, p_cln, label="denoised", alpha=0.8)
ax.axvline(60, ls="--", c="red", alpha=0.5, label="60 Hz powerline")
ax.set_xlim(0, 100)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power spectral density")
ax.set_title("Power spectrum before and after filtering")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "power_spectrum.png", dpi=120)
plt.close()

snr_improvement = 10 * np.log10(np.var(clean_sig) / np.var(raw_sig - clean_sig))
print(f"Estimated SNR after filtering: {snr_improvement:.2f} dB")

## D4. Exploratory Data Analysis

In [ ]:
summary = []
for rid in ACTIVE_RECORDS:
    try:
        r, a = load_record(rid)
        symbols = [s for s in a.symbol if s in AAMI_MAP]
        classes = [AAMI_MAP[s] for s in symbols]
        rr = np.diff(a.sample) / FS
        rr = rr[(rr > 0.2) & (rr < 2.0)]          # physiological plausibility filter
        summary.append({
            "record": rid,
            "duration_min": round(r.p_signal.shape[0] / r.fs / 60, 1),
            "n_beats": len(symbols),
            "mean_hr": round(60 / rr.mean(), 1) if len(rr) else np.nan,
            "hrv_sdnn_ms": round(rr.std() * 1000, 1) if len(rr) else np.nan,
            **Counter(classes),
        })
    except Exception as exc:
        print(f"  skipped {rid}: {exc}")

summary_df = pd.DataFrame(summary).fillna(0)
print("Per-record summary:\n", summary_df.to_string(index=False))

total_by_class = {c: int(summary_df.get(c, pd.Series([0])).sum()) for c in AAMI_CLASSES}
print("\nTotal beats by AAMI class:", total_by_class)
nonzero = {k: v for k, v in total_by_class.items() if v}
print(f"Imbalance ratio: {max(nonzero.values()) / min(nonzero.values()):.1f} : 1")
print("This extreme skew is inherent - most heartbeats in any recording are normal.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
pd.Series(total_by_class).plot.bar(ax=axes[0], rot=0, logy=True)
axes[0].set_title("Beat count by AAMI class (log scale)")
sns.histplot(summary_df["mean_hr"], bins=15, ax=axes[1])
axes[1].set_title("Mean heart rate across records")
sns.histplot(summary_df["hrv_sdnn_ms"], bins=15, ax=axes[2])
axes[2].set_title("Heart rate variability (SDNN)")
plt.tight_layout()
plt.savefig(OUT_DIR / "signal_eda.png", dpi=120)
plt.close()

## D5. Beat segmentation into fixed windows

A classifier needs equal-length inputs, but heartbeats vary in duration. The
standard approach is to centre a fixed window on each annotated R peak:
250 ms before (captures the P wave and PR interval) and 450 ms after (captures
the QRS complex, ST segment and T wave).

Each beat is z-score normalised individually, which removes per-patient
amplitude differences caused by electrode placement and body habitus.

In [ ]:
def segment_beats(rec_id: str, lead: int = 0):
    """Extract fixed-length, normalised beat windows from one record."""
    r, a = load_record(rec_id)
    sig = denoise_ecg(r.p_signal[:, lead])

    before = int(WINDOW_BEFORE * FS)
    after = int(WINDOW_AFTER * FS)

    beats, labels, rr_features = [], [], []
    peaks = a.sample
    for i, (peak, sym) in enumerate(zip(peaks, a.symbol)):
        if sym not in AAMI_MAP:
            continue                                    # not a beat annotation
        start, end = peak - before, peak + after
        if start < 0 or end > len(sig):
            continue                                    # window falls off the edge

        beat = sig[start:end]
        sd = beat.std()
        if sd < 1e-6:
            continue                                    # flatline segment
        beat = (beat - beat.mean()) / sd                # per-beat z-score

        # RR-interval context - rhythm irregularity is diagnostic on its own,
        # and the morphology window alone cannot capture it.
        prev_rr = (peak - peaks[i - 1]) / FS if i > 0 else np.nan
        next_rr = (peaks[i + 1] - peak) / FS if i < len(peaks) - 1 else np.nan
        local_rr = np.diff(peaks[max(0, i - 10):i + 1]).mean() / FS if i >= 1 else np.nan

        beats.append(beat)
        labels.append(AAMI_MAP[sym])
        rr_features.append([prev_rr, next_rr,
                            prev_rr / local_rr if local_rr else np.nan,
                            local_rr])

    return (np.array(beats, dtype=np.float32),
            np.array(labels),
            np.array(rr_features, dtype=np.float32))


all_beats, all_labels, all_rr = [], [], []
for rid in ACTIVE_RECORDS:
    try:
        b, l, r_ = segment_beats(rid)
        all_beats.append(b)
        all_labels.append(l)
        all_rr.append(r_)
        print(f"  {rid}: {len(b)} beats  {dict(Counter(l))}")
    except Exception as exc:
        print(f"  {rid} failed: {exc}")

X_beats = np.vstack(all_beats)
y_labels = np.concatenate(all_labels)
rr_feats = np.vstack(all_rr)
rr_feats = np.nan_to_num(rr_feats, nan=np.nanmedian(rr_feats))

print(f"\nSegmented dataset: {X_beats.shape}  ({WIN_LEN} samples per beat)")
print("Class counts:", dict(Counter(y_labels)))

In [ ]:
# Average beat morphology per class - a sanity check that segmentation worked
fig, axes = plt.subplots(1, len(AAMI_CLASSES), figsize=(19, 3.4), sharey=True)
t_beat = (np.arange(X_beats.shape[1]) - WINDOW_BEFORE * FS) / FS * 1000
for ax, cls in zip(axes, AAMI_CLASSES):
    mask = y_labels == cls
    if mask.sum() == 0:
        ax.set_title(f"{cls} (none)")
        ax.axis("off")
        continue
    mean_beat = X_beats[mask].mean(axis=0)
    sd_beat = X_beats[mask].std(axis=0)
    ax.plot(t_beat, mean_beat, color="tab:blue")
    ax.fill_between(t_beat, mean_beat - sd_beat, mean_beat + sd_beat, alpha=0.25)
    ax.axvline(0, ls="--", c="red", alpha=0.6)
    ax.set_title(f"Class {cls}  (n={mask.sum()})")
    ax.set_xlabel("ms from R peak")
axes[0].set_ylabel("normalised amplitude")
plt.suptitle("Mean beat morphology by AAMI class (shaded = 1 SD)")
plt.tight_layout()
plt.savefig(OUT_DIR / "beat_morphology.png", dpi=120)
plt.close()

## D6. Feature extraction

Two parallel representations: the raw waveform for a 1D CNN, and hand-crafted
time/frequency features for classical models.

In [ ]:
def extract_beat_features(beat: np.ndarray, fs: int = FS) -> dict:
    """Statistical, morphological and spectral descriptors of one beat."""
    r_idx = int(WINDOW_BEFORE * fs)
    freqs, psd = sps.welch(beat, fs, nperseg=min(128, len(beat)))
    psd_norm = psd / (psd.sum() + 1e-12)

    return {
        # Time-domain statistics
        "mean": beat.mean(), "std": beat.std(),
        "skew": stats.skew(beat), "kurtosis": stats.kurtosis(beat),
        "rms": np.sqrt((beat ** 2).mean()),
        "ptp": beat.ptp(),
        # Morphology
        "r_amplitude": beat[r_idx],
        "q_amplitude": beat[max(0, r_idx - 20):r_idx].min(),
        "s_amplitude": beat[r_idx:r_idx + 20].min(),
        "t_amplitude": beat[r_idx + 40:].max(),
        "qrs_energy": float((beat[r_idx - 20:r_idx + 20] ** 2).sum()),
        # Complexity
        "zero_crossings": int(np.sum(np.diff(np.sign(beat)) != 0)),
        "signal_energy": float((beat ** 2).sum()),
        # Frequency domain
        "spectral_centroid": float((freqs * psd_norm).sum()),
        "spectral_entropy": float(-(psd_norm * np.log(psd_norm + 1e-12)).sum()),
        "dominant_freq": float(freqs[psd.argmax()]),
    }


print("Extracting handcrafted features...")
feat_rows = [extract_beat_features(b) for b in X_beats]
feat_df = pd.DataFrame(feat_rows)
feat_df[["prev_rr", "next_rr", "rr_ratio", "local_rr"]] = rr_feats
feat_df["label"] = y_labels

print("Feature matrix:", feat_df.shape)
print(feat_df.groupby("label")[
    ["r_amplitude", "qrs_energy", "prev_rr", "spectral_entropy"]
].mean().round(3))

## D7. Augmentation for minority classes

S and F beats are rare, so the minority classes are augmented with
transformations that preserve diagnostic morphology: amplitude scaling
(electrode contact variance), Gaussian noise (recording conditions), and small
time warps (heart rate variation). Beats are never flipped or reversed - the
direction of a QRS complex is diagnostic.

In [ ]:
def augment_beat(beat: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = beat.copy()
    out = out * rng.uniform(0.85, 1.15)                     # amplitude scaling
    out = out + rng.normal(0, 0.03, size=out.shape)         # additive noise
    shift = rng.integers(-8, 9)                             # small temporal shift
    out = np.roll(out, shift)
    # Mild time warp via resample-and-crop
    factor = rng.uniform(0.94, 1.06)
    warped = sps.resample(out, int(len(out) * factor))
    if len(warped) >= len(out):
        start = (len(warped) - len(out)) // 2
        out = warped[start:start + len(out)]
    else:
        pad = len(out) - len(warped)
        out = np.pad(warped, (pad // 2, pad - pad // 2), mode="edge")
    return out.astype(np.float32)


rng = np.random.default_rng(RANDOM_STATE)
counts = Counter(y_labels)
target = int(np.median([v for v in counts.values()]))

aug_beats, aug_labels = [], []
for cls in AAMI_CLASSES:
    idx = np.where(y_labels == cls)[0]
    if len(idx) == 0 or len(idx) >= target:
        continue
    need = target - len(idx)
    picks = rng.choice(idx, size=need, replace=True)
    for p in picks:
        aug_beats.append(augment_beat(X_beats[p], rng))
        aug_labels.append(cls)
    print(f"  class {cls}: {len(idx)} -> {len(idx) + need} (+{need} synthetic)")

if aug_beats:
    X_aug = np.vstack([X_beats, np.array(aug_beats, dtype=np.float32)])
    y_aug = np.concatenate([y_labels, np.array(aug_labels)])
else:
    X_aug, y_aug = X_beats, y_labels

print("\nAfter augmentation:", X_aug.shape, dict(Counter(y_aug)))

## D8. Model-ready signal tensors

Splitting is done **by record**, not by beat. Beats from one patient are highly
correlated, so a random beat-level split leaks patient-specific morphology into
the test set and inflates accuracy dramatically. Record-level splitting is the
honest evaluation protocol - the inter-patient paradigm in the literature.

In [ ]:
from sklearn.preprocessing import LabelEncoder  # noqa: E402

le = LabelEncoder().fit(AAMI_CLASSES)

# Record-level split
split_at = int(len(ACTIVE_RECORDS) * 0.75)
train_recs, test_recs = ACTIVE_RECORDS[:split_at], ACTIVE_RECORDS[split_at:]
print("Train records:", train_recs)
print("Test records: ", test_recs)


def build_split(records):
    Xs, ys = [], []
    for rid in records:
        try:
            b, l, _ = segment_beats(rid)
            Xs.append(b)
            ys.append(l)
        except Exception:
            continue
    return np.vstack(Xs), np.concatenate(ys)


X_tr, y_tr = build_split(train_recs)
X_te, y_te = build_split(test_recs)

X_tr = X_tr[..., np.newaxis]     # (N, WIN_LEN, 1) for a 1D CNN
X_te = X_te[..., np.newaxis]
y_tr_enc = le.transform(y_tr)
y_te_enc = le.transform(y_te)

print("\nTrain tensor:", X_tr.shape, " Test tensor:", X_te.shape)
print("Train classes:", dict(Counter(y_tr)))
print("Test classes: ", dict(Counter(y_te)))

cls_counts = Counter(y_tr_enc)
class_weights = {int(c): len(y_tr_enc) / (len(cls_counts) * n)
                 for c, n in cls_counts.items()}
print("Class weights:", {k: round(v, 2) for k, v in class_weights.items()})

np.savez_compressed(
    OUT_DIR / "mitbih_model_ready.npz",
    X_train=X_tr, y_train=y_tr_enc,
    X_test=X_te, y_test=y_te_enc,
    class_names=le.classes_,
    window_length=WIN_LEN, fs=FS,
)
feat_df.to_csv(OUT_DIR / "mitbih_handcrafted_features.csv", index=False)
print(f"\nSaved to {OUT_DIR}")

## Part D summary

| Step | Finding |
|---|---|
| Noise removal | Median-filter baseline removal + 60 Hz notch + 0.5-40 Hz bandpass, all zero-phase |
| Segmentation | 252-sample windows (-250 ms / +450 ms around each annotated R peak) |
| Labels | 19 MIT-BIH symbols collapsed to 5 AAMI superclasses |
| Imbalance | Severe and inherent - normal beats dominate by orders of magnitude |
| Features | Raw waveform tensor + 19 handcrafted time/frequency/morphology descriptors |
| Split | By record, not by beat, to prevent patient-level leakage |